# 163 — Logit Gradient Analysis

Analyze how logits depend on activations using gradients, Jacobian structure,
per-dimension impact, and curvature to understand prediction sensitivity.

## Why This Matters

Gradient logit traces how training signals and attribution scores flow through the network. Understanding gradient structure reveals which components are most trainable, which carry the strongest attribution signals, and how LayerNorm affects learning.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.logit_gradient_analysis import (
    logit_sensitivity_profile,
    logit_jacobian_structure,
    per_dimension_logit_impact,
    gradient_alignment_across_layers,
    logit_curvature,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.arange(15)

In [ ]:
result = logit_sensitivity_profile(model, tokens)
print(f"Target token: {result['target_token']}\n")
for p in result['per_layer']:
    bar = '#' * int(p['relative_sensitivity'] * 30)
    print(f"Layer {p['layer']}: sens={p['sensitivity']:.4f}  "
          f"rel={p['relative_sensitivity']:.3f}  {bar}")

In [ ]:
result = logit_jacobian_structure(model, tokens)
print(f"Effective rank: {result['effective_rank']:.1f} / {result['total_singular_values']}")
print(f"Condition number: {result['condition_number']:.0f}")
print(f"Top-5 concentration: {result['concentration']:.3f}")
print(f"Top singular values: {result['top_singular_values']}")

In [ ]:
result = per_dimension_logit_impact(model, tokens, top_k=10)
print(f"Target token: {result['target_token']}")
print(f"Top-10 concentration: {result['concentration']:.3f}\n")
for d in result['top_dimensions']:
    bar = '#' * int(abs(d['impact']) * 50)
    sign = '+' if d['impact'] > 0 else '-'
    print(f"  dim {d['dimension']:3d}: {sign}{d['abs_impact']:.4f}  {bar}")

In [ ]:
result = gradient_alignment_across_layers(model, tokens)
print(f"Mean alignment: {result['mean_alignment']:.4f}\n")
for p in result['per_layer']:
    print(f"Layer {p['layer']}: grad_norm={p['gradient_norm']:.4f}")

In [ ]:
result = logit_curvature(model, tokens)
for p in result['per_layer']:
    bar = '#' * min(int(p['curvature'] * 10), 40)
    print(f"Layer {p['layer']}: curvature={p['curvature']:.4f}  base_logit={p['base_logit']:.4f}  {bar}")